In [155]:
import json
import time
import hashlib

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

In [156]:
BASE_DIR = Path("akomodasi_kintamani_2501010378")

FOLDER_LIST = [
    BASE_DIR / "data" / "raw",
    BASE_DIR / "data" / "interim",
    BASE_DIR / "notebooks",
    BASE_DIR / "docs",
    BASE_DIR / "reports"
]


for folder in FOLDER_LIST:
    folder.mkdir(parents=True, exist_ok=True)

print(f" Sel 2 Berhasil: Folder utama '{BASE_DIR}' beserta seluruh sub-foldernya telah dibuat!")

 Sel 2 Berhasil: Folder utama 'akomodasi_kintamani_2501010378' beserta seluruh sub-foldernya telah dibuat!


In [157]:
URL = "https://overpass-api.de/api/interpreter"

HEADERS = {
    "User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (2501010378@instiki.ac.id)"
}

QUERY = """
[out:json][timeout:90];

area["name"="Kintamani"]
["boundary"="administrative"]
["admin_level"="6"]->.kec;

nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"]
(area.kec);

out center tags;
"""


In [158]:
def ambil_overpass(query, maks_percobaan=4, jeda=5):

    for percobaan in range(1, maks_percobaan + 1):

        try:
            r = requests.post(
                URL,
                data={"data": query},
                headers=HEADERS,
                timeout=120
            )

            if r.status_code == 200:
                return r.json()

            print(
                f"[percobaan {percobaan}] "
                f"status {r.status_code}, tunggu {jeda}s"
            )

        except (requests.Timeout, requests.ConnectionError) as e:

            print(
                f"[percobaan {percobaan}] "
                f"{type(e).__name__}, tunggu {jeda}s"
            )

        time.sleep(jeda)
        jeda *= 2

    raise RuntimeError("Gagal mengambil data dari Overpass")

In [140]:
data = ambil_overpass(QUERY)

print("Jumlah elemen:", len(data["elements"]))

[percobaan 1] status 502, tunggu 5s
Jumlah elemen: 32


In [141]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

Path("data/raw").mkdir(parents=True, exist_ok=True)

raw_path = Path(
    f"data/raw/osm_akomodasi_kintamani_{stamp}.json"
)

raw_path.write_text(
    json.dumps(data, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Data tersimpan:", raw_path)

Data tersimpan: data/raw/osm_akomodasi_kintamani_20261111_065100.json


In [142]:
md5_hash = hashlib.md5(
    raw_path.read_bytes()
).hexdigest()

provenance = {
    "url_endpoint": URL,
    "query": QUERY,
    "waktu_akuisisi_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "jumlah_elemen": len(data["elements"]),
    "nama_file_mentah": raw_path.name,
    "hash_md5": md5_hash,
    "lisensi": "ODbL",
    "atribusi": "© OpenStreetMap contributors",
    "nama_pengambil": "Anak Agung Radha Noviyanti",
    "nim_pengambil": "2501010378"
}

Path("docs").mkdir(parents=True, exist_ok=True)

Path("docs/provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Provenance berhasil dibuat!")
print(provenance)

Provenance berhasil dibuat!
{'url_endpoint': 'https://overpass-api.de/api/interpreter', 'query': '\n[out:json][timeout:90];\n\narea["name"="Kintamani"]\n["boundary"="administrative"]\n["admin_level"="6"]->.kec;\n\nnwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"]\n(area.kec);\n\nout center tags;\n', 'waktu_akuisisi_utc': '2026-11-11T06:51:00.308327+00:00', 'jumlah_elemen': 32, 'nama_file_mentah': 'osm_akomodasi_kintamani_20261111_065100.json', 'hash_md5': '85d2cbc346cb2008d9285149bbc8b425', 'lisensi': 'ODbL', 'atribusi': '© OpenStreetMap contributors', 'nama_pengambil': 'Anak Agung Radha Noviyanti', 'nim_pengambil': '2501010378'}


In [143]:
with open(raw_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Jumlah elemen:", len(data["elements"]))

Jumlah elemen: 32


In [144]:
import pandas as pd
from IPython.display import display

def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)
        baris.append({
            "osm_type": e["type"], "osm_id": e["id"], "nama": tg.get("name"),
            "jenis": tg.get("tourism"), "lat": pusat.get("lat"), "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"), "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"), "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}"
        })
    return pd.DataFrame(baris)

akomodasi = ratakan(data["elements"])
akomodasi.to_csv("data/interim/akomodasi_kintamani.csv", index=False, encoding="utf-8-sig")

print("--- Pemeriksaan Dasar ---")
print("Jumlah duplikat:", akomodasi.duplicated(subset=["osm_type", "osm_id"]).sum())
print("Tanpa nama:", akomodasi["nama"].isna().sum())
print("\nCuplikan Tabel Akomodasi:")
display(akomodasi.head())

--- Pemeriksaan Dasar ---
Jumlah duplikat: 0
Tanpa nama: 1

Cuplikan Tabel Akomodasi:


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
0,node,1977779907,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,None,None,None,http://lakeviewbali.com/en/,https://www.openstreetmap.org/node/1977779907
1,node,2107226701,Arlina,guest_house,-8.249815,115.398465,None,None,None,None,https://www.openstreetmap.org/node/2107226701
2,node,2107227001,Toya Devasya Resort,hotel,-8.250164,115.399798,None,None,None,None,https://www.openstreetmap.org/node/2107227001
3,node,4150326329,Guesthouse,hotel,-8.281456,115.362223,None,None,None,None,https://www.openstreetmap.org/node/4150326329
4,node,4297310696,The Cave Hotel,hotel,-8.280748,115.362048,None,2,5,None,https://www.openstreetmap.org/node/4297310696


In [145]:
akomodasi = ratakan(data["elements"])

print(akomodasi.head())

  osm_type      osm_id                     nama        jenis       lat  \
0     node  1977779907  The Lakeview Hotel Bali        hotel -8.283140   
1     node  2107226701                   Arlina  guest_house -8.249815   
2     node  2107227001      Toya Devasya Resort        hotel -8.250164   
3     node  4150326329               Guesthouse        hotel -8.281456   
4     node  4297310696           The Cave Hotel        hotel -8.280748   

          lon  desa bintang jumlah_kamar                      website  \
0  115.365844  None    None         None  http://lakeviewbali.com/en/   
1  115.398465  None    None         None                         None   
2  115.399798  None    None         None                         None   
3  115.362223  None    None         None                         None   
4  115.362048  None       2            5                         None   

                                         url_osm  
0  https://www.openstreetmap.org/node/1977779907  
1  https://www

In [146]:
print(akomodasi["jenis"].value_counts())

jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64


In [147]:
print(
    "Tanpa nama:",
    akomodasi["nama"].isna().sum()
)

Tanpa nama: 1


In [148]:
Path("data/interim").mkdir(
    parents=True,
    exist_ok=True
)

akomodasi.to_csv(
    "data/interim/akomodasi_kintamani.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV berhasil disimpan!")

CSV berhasil disimpan!


In [149]:
duplikat = akomodasi.duplicated(
    subset=["osm_type", "osm_id"]
).sum()

print("Jumlah duplikat:", duplikat)

Jumlah duplikat: 0


In [150]:
print("Latitude kosong:", akomodasi["lat"].isna().sum())
print("Longitude kosong:", akomodasi["lon"].isna().sum())

Latitude kosong: 0
Longitude kosong: 0


In [151]:
print(akomodasi.isna().sum())

osm_type         0
osm_id           0
nama             1
jenis            0
lat              0
lon              0
desa            32
bintang         31
jumlah_kamar    30
website         30
url_osm          0
dtype: int64


In [162]:
print("--- 10 Sampel Acak untuk Diverifikasi Manual ---")
sampel_verifikasi = akomodasi.sample(n=10, random_state=42)
display(sampel_verifikasi[['nama', 'jenis', 'lat', 'lon']])

print("\n--- Tabel Perbandingan BPS ---")
df_bps = pd.DataFrame({
    "sumber_data": ["OSM (Overpass API)", "BPS Kabupaten Bangli"],
    "tahun_data": ["2026", "2025"],
    "jumlah_akomodasi": [len(akomodasi), "32"],
    "keterangan": ["Data ditarik otomatis dari API", "Data dari publikasi Kintamani Dalam Angka"]
})
df_bps.to_csv("docs/perbandingan_bps.csv", index=False, encoding="utf-8-sig")
display(df_bps)

--- 10 Sampel Acak untuk Diverifikasi Manual ---


,nama,jenis,lat,lon
29,Batur Sunrise Guesthouse,hotel,-8.249020,115.400863
15,Caldera,hotel,-8.279139,115.367364
24,Bunbulan Panorama Villas,guest_house,-8.222736,115.397971
17,Segara Hotel,hotel,-8.278086,115.379134
8,The Volcania Guesthouse,hotel,-8.283796,115.365242
9,Entrance to: Bunbulan Hill Hostel,hostel,-8.222014,115.395349
30,Lava Bali and Hot Spring,hotel,-8.250311,115.396374
25,Villa Jempana,hotel,-8.231158,115.410556
12,Hut,chalet,-8.242693,115.387587
0,The Lakeview Hotel Bali,hotel,-8.283140,115.365844



--- Tabel Perbandingan BPS ---


,sumber_data,tahun_data,jumlah_akomodasi,keterangan
0,OSM (Overpass API),2026,32,Data ditarik otomatis dari API
1,BPS Kabupaten Bangli,2025,32,Data dari publikasi Kintamani Dalam Angka


In [161]:
# Membuat Data Dictionary
df_dict = pd.DataFrame([
    {"kolom": "osm_type", "tipe": "teks", "deskripsi": "Jenis objek OSM (node, way, relation)", "contoh": "node", "boleh_kosong": "Tidak"},
    {"kolom": "osm_id", "tipe": "bilangan bulat", "deskripsi": "ID objek OpenStreetMap", "contoh": "2107227001", "boleh_kosong": "Tidak"},
    {"kolom": "nama", "tipe": "teks", "deskripsi": "Nama akomodasi", "contoh": "Toya Devasya Resort", "boleh_kosong": "Ya"},
    {"kolom": "jenis", "tipe": "teks", "deskripsi": "Nilai tag tourism", "contoh": "hotel", "boleh_kosong": "Tidak"},
    {"kolom": "lat", "tipe": "desimal", "deskripsi": "Koordinat latitude", "contoh": "-8.250164", "boleh_kosong": "Tidak"},
    {"kolom": "lon", "tipe": "desimal", "deskripsi": "Koordinat longitude", "contoh": "115.399798", "boleh_kosong": "Tidak"},
    {"kolom": "desa", "tipe": "teks", "deskripsi": "Desa sesuai tag addr:village", "contoh": "Kedisan", "boleh_kosong": "Ya"},
    {"kolom": "bintang", "tipe": "teks", "deskripsi": "Klasifikasi bintang", "contoh": "3", "boleh_kosong": "Ya"},
    {"kolom": "jumlah_kamar", "tipe": "teks", "deskripsi": "Jumlah kamar", "contoh": "20", "boleh_kosong": "Ya"},
    {"kolom": "website", "tipe": "teks", "deskripsi": "Situs resmi", "contoh": "http://lakeviewbali.com", "boleh_kosong": "Ya"},
    {"kolom": "url_osm", "tipe": "teks", "deskripsi": "Tautan OSM", "contoh": "https://www.openstreetmap.org/node/2107227001", "boleh_kosong": "Tidak"}
])
df_dict.to_csv("docs/data_dictionary.csv", index=False, encoding="utf-8-sig")

# Membuat Data Ethics Checklist
df_ethics = pd.DataFrame([
    {"pertanyaan_etika": "Sumber data diizinkan?", "status": "Ya", "bukti_atau_keterangan": "Overpass API dipakai tanpa scraping platform OTA."},
    {"pertanyaan_etika": "Patuhi rate limit?", "status": "Ya", "bukti_atau_keterangan": "Fungsi akuisisi memakai time.sleep dan exponential backoff."},
    {"pertanyaan_etika": "Data kontak pribadi?", "status": "Tidak", "bukti_atau_keterangan": "Nomor telepon & email diabaikan saat parse tag OSM."},
    {"pertanyaan_etika": "Atribusi dicantumkan?", "status": "Ya", "bukti_atau_keterangan": "© OpenStreetMap contributors ada di JSON dan README."}
])
df_ethics.to_csv("docs/data_ethics_checklist.csv", index=False, encoding="utf-8-sig")

# Membuat file README.md
Path("README.md").write_text("# Akuisisi Data Akomodasi Kintamani\n\n**Sumber Data & Atribusi:**\nData diperoleh dari OpenStreetMap melalui Overpass API. **© OpenStreetMap contributors**. Data disediakan di bawah lisensi ODbL.\n\nSemua file CSV hasil proses telah diekspor ke folder `docs` dan `data/interim`.", encoding="utf-8")

print("\n--- Kamus Data ---")
display(df_dict.head(3))
print("\n--- Data Ethics Checklist ---")
display(df_ethics)


--- Kamus Data ---


,kolom,tipe,deskripsi,contoh,boleh_kosong
0,osm_type,teks,"Jenis objek OSM (node, way, relation)",node,Tidak
1,osm_id,bilangan bulat,ID objek OpenStreetMap,2107227001,Tidak
2,nama,teks,Nama akomodasi,Toya Devasya Resort,Ya



--- Data Ethics Checklist ---


,pertanyaan_etika,status,bukti_atau_keterangan
0,Sumber data diizinkan?,Ya,Overpass API dipakai tanpa scraping platform OTA.
1,Patuhi rate limit?,Ya,Fungsi akuisisi memakai time.sleep dan exponen...
2,Data kontak pribadi?,Tidak,Nomor telepon & email diabaikan saat parse tag...
3,Atribusi dicantumkan?,Ya,© OpenStreetMap contributors ada di JSON dan R...


In [152]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [163]:
import shutil
from google.colab import files

# Membungkus foldernya menjadi zip agar mudah diunduh
shutil.make_archive('data_kintamani', 'zip', 'data')
shutil.make_archive('docs_kintamani', 'zip', 'docs')

print("Seluruh folder siap! Mengunduh berkas...")
files.download('data_kintamani.zip')
files.download('docs_kintamani.zip')
files.download('README.md')

Seluruh folder siap! Mengunduh berkas...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>